# Personne B — Saisonnalité et risque de retard (J3 §3 + table du simulateur, J4 test n°2, onglet « Saisonnalité & risque »)

**Questions** : quand les vols sont-ils en retard ? Quel est le risque pour un vol donné (compagnie, aéroport, mois, créneau) ?

| Livrable | Où |
|---|---|
| J3 §3 : table Gold `monthly_trend` (série mensuelle + `LAG`) | `/lakehouse/gold/monthly_trend` |
| J3 : table Gold `delay_risk` (base du simulateur voyageur) | `/lakehouse/gold/delay_risk` |
| J4 test n°2 : χ² entre créneau horaire et retard | ce notebook |
| Onglet « Saisonnalité & risque » du dashboard (simulateur) | `app/sections/temps_risque.py` |

**Règles communes** (détail dans `docs/repartition_J3_J4.md`)
- Ne modifiez que ce notebook et votre fichier `app/sections/temps_risque.py` : aucun conflit git possible.
- Les tables Gold respectent **exactement** les noms et colonnes du contrat : le dashboard et l'assemblage final en dépendent. `save_gold(...)` le vérifie.
- Taux de retard = part des vols **arrivés** avec `arrival_delay >= 15` (vue `v_arrives`). Taux d'annulation = sur **tous** les vols (`v_silver`).
- Octobre a des codes aéroport non valides : pour toute analyse par aéroport, filtrez `airport_code_valid`.
- Chaque résultat est suivi d'une interprétation en Markdown (comme en J2). Committez le notebook **exécuté**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("B_temps_risque").getOrCreate()
sns.set_theme(style="whitegrid", context="notebook")

SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
# Vols arrivés : les seuls qui ont un arrival_delay (même périmètre que l'EDA de J2)
spark.sql("""
    SELECT * FROM v_silver
    WHERE NOT is_cancelled AND NOT is_diverted AND arrival_delay IS NOT NULL
""").createOrReplaceTempView("v_arrives")
print("v_silver :", spark.table("v_silver").count(), "vols | v_arrives :", spark.table("v_arrives").count(), "vols arrivés")


def save_gold(df, name, contrat, n_files=1):
    """Vérifie les colonnes du contrat (docs/repartition_J3_J4.md) puis écrit /lakehouse/gold/<name> en Delta."""
    manquantes = [c for c in contrat if c not in df.columns]
    assert not manquantes, f"Colonnes du contrat manquantes pour {name} : {manquantes}"
    path = f"/lakehouse/gold/{name}"
    autres = [c for c in df.columns if c not in contrat]
    (df.select(*contrat, *autres).coalesce(n_files)
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path))
    n = spark.read.format("delta").load(path).count()
    print(f"Table Gold écrite : {path} ({n:,} lignes)")

## J3 §3 — Table Gold `monthly_trend` : dynamique mensuelle avec `LAG()`

Contrat (12 lignes, une par mois) :

| colonne | définition |
|---|---|
| `month` | 1 à 12 |
| `n_vols` | vols arrivés |
| `taux_retard` | % de vols arrivés avec `arrival_delay >= 15` |
| `retard_median` | médiane de `arrival_delay` |
| `taux_annulation` | % de vols annulés (sur `v_silver`, tous les vols) |
| `taux_retard_mois_prec` | `LAG(taux_retard, 1) OVER (ORDER BY month)` |
| `evolution_pts` | `taux_retard - taux_retard_mois_prec` (points de %) |
| `moyenne_glissante_3m` | `AVG(taux_retard) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` |

Astuce : deux CTE (`retards` sur `v_arrives`, `annulations` sur `v_silver`) joints sur `month`, puis les fenêtres.

In [ ]:
MONTHLY_TREND_COLS = ["month", "n_vols", "taux_retard", "retard_median", "taux_annulation",
                      "taux_retard_mois_prec", "evolution_pts", "moyenne_glissante_3m"]

df_gold_2 = spark.sql("""
    WITH retards AS (
        SELECT ...   -- TODO
    ),
    annulations AS (
        SELECT ...   -- TODO
    )
    SELECT ...       -- TODO : jointure + LAG + moyenne glissante
    ORDER BY month
""")
save_gold(df_gold_2, "monthly_trend", MONTHLY_TREND_COLS)
df_gold_2.show(12, truncate=False)

**Interprétation** *(à compléter)* : pics de février et juin, creux d'octobre (cf. conclusion J2) ; que montre la moyenne glissante que le mois brut ne montre pas ?

## J3 — Table Gold `delay_risk` : base du simulateur voyageur

Une ligne par combinaison (compagnie, aéroport de départ, mois, créneau). Le dashboard filtre cette table et calcule `sum(n_retards) / sum(n_vols)`.

| colonne | définition |
|---|---|
| `airline`, `airline_name` | compagnie |
| `origin_airport` | `CASE WHEN airport_code_valid THEN origin_airport END` (NULL en octobre : codes non valides) |
| `month` | 1 à 12 |
| `dep_slot` | créneau de `dep_hour` (heure prévue) : `1_nuit (0-5h)`, `2_matin (6-11h)`, `3_apres-midi (12-16h)`, `4_soir (17-23h)` |
| `n_vols`, `n_retards`, `taux_retard` | comme `airline_ranking` |

On garde les combinaisons rares (le dashboard additionne) : ne filtrez pas sur `n_vols`.

In [ ]:
DELAY_RISK_COLS = ["airline", "airline_name", "origin_airport", "month", "dep_slot", "n_vols", "n_retards", "taux_retard"]

df_risk = spark.sql("""
    SELECT ...   -- TODO
    FROM v_arrives
    GROUP BY ...
""")
save_gold(df_risk, "delay_risk", DELAY_RISK_COLS, n_files=2)

# Contrôle : la somme des n_vols doit égaler le nombre de vols arrivés
assert df_risk.agg(F.sum("n_vols")).first()[0] == spark.table("v_arrives").count()
df_risk.orderBy(F.desc("n_vols")).show(10, truncate=False)

## J4 test n°2 — χ² : le créneau horaire et le retard sont-ils liés ?

- $H_0$ : être en retard (≥ 15 min) est indépendant du créneau de départ. $H_1$ : il y a un lien.
- Pourquoi le χ² : deux variables qualitatives (créneau, en retard oui/non). Le tableau de contingence se calcule **dans Spark** (`groupBy("dep_slot", "is_delayed_15").count()`), seules 8 cases reviennent en pandas.
- Vérifiez que tous les effectifs attendus sont ≥ 5 et ajoutez le **V de Cramér** comme taille d'effet.

In [ ]:
# TODO : tableau de contingence créneau × retard, stats.chi2_contingency, V de Cramér
contingence = ...
chi2, p_value, ddl, attendus = ...
print(f"Chi² = {chi2:.1f}, ddl = {ddl}, p = {p_value:.2e}, effectif attendu minimal = {attendus.min():.0f}")

**Conclusion du test** *(à compléter)*.